In [1]:
import json
import numpy as np
import pandas as pd

In [2]:
params_json = json.load(open("../conf/parameters.json"))
print(params_json.keys())
data = pd.read_csv("../data/L0123001.txt", sep="\t", parse_dates=["Time"], index_col="Time")
data

dict_keys(['KC', 'UM', 'LM', 'C', 'WM', 'B', 'IM', 'SM', 'EX', 'KG', 'KI', 'CS', 'CI', 'CG', 'CR', 'KE', 'XE'])


,P,T,E,Qls,Qmm
Time,,,,,
1984-01-01,4.1,0.5,0.2,2640.0,0.6336
1984-01-02,15.9,0.2,0.2,3440.0,0.8256
1984-01-03,0.8,0.9,0.3,12200.0,2.9280
1984-01-04,0.0,0.5,0.3,7600.0,1.8240
1984-01-05,0.0,-1.6,0.1,6250.0,1.5000
...,...,...,...,...,...
2012-12-27,5.1,6.3,0.5,2442.0,0.5861
2012-12-28,0.0,1.6,0.3,2983.0,0.7159
2012-12-29,0.6,0.4,0.2,2943.0,0.7063


In [3]:
# 蒸发参数
KC = params_json["KC"]["default"]
UM = params_json["UM"]["default"]
LM = params_json["LM"]["default"]
C  = params_json["C"]["default"]
# 产流参数
WM = params_json["WM"]["default"]
B  = params_json["B"]["default"]
IM = params_json["IM"]["default"]
# 分水源参数
SM = params_json["SM"]["default"]
EX = params_json["EX"]["default"]
KG = params_json["KG"]["default"]
KI = params_json["KI"]["default"]
# 汇流参数（坡面汇流，线性水库）
CS = params_json["CS"]["default"]
CI = params_json["CI"]["default"]
CG = params_json["CG"]["default"]
# 汇流参数（河网汇流，线性水库）
CR = params_json["CR"]["default"]
# 汇流参数（河道汇流，马斯京根法）
KE = params_json["KE"]["default"]
XE = params_json["XE"]["default"]

params = [KC, UM, LM, C, WM, B, IM, SM, EX, KG, KI, CS, CI, CG, CR, KE, XE]
data   = data[['P', 'T', 'E']].values

In [ ]:
def xinanjiang(data:       np.ndarray,
               params:     np.ndarray,
               init_state: np.ndarray):
    """
    Xinanjiang model implementation.
    """
    # data: (n, 3)
    # ------------
    # data[:, 0]  # 降水
    # data[:, 1]  # 气温
    # data[:, 2]  # 潜在蒸散发
    # 获取参数
    KC, UM, LM, C, WM, B, IM, SM, EX, KG, KI, CS, CI, CG, CR, KE, XE = params
    # 降水、气温、潜在蒸散发数据
    P, T, E = data
    # 时间步初始三层土壤含水量
    WU, WL, WD = init_state
    # 时间步开始时的总土壤含水量
    W = WU + WL + WD

    # 深层土壤蓄水容量
    DM = WM - UM - LM
    # 流域单点最大土壤含水量
    WMM = (1 + B) * WM
    # 流域单点最大自由水含水量
    SMM = (1 + EX) * SM

    # 流域初始状态
    A0 = WMM * (1 - np.power(1 - W / WMM, 1 / (1 + B)))
    S0 = 0
    AU = 0
    FR = 1 - np.power(1 - A0 / WMM, B)

    #### 三层蒸散发

    # 流域蒸发能力
    EP = KC * E
    if P + WU >= EP:
        EU = EP
        EL = 0
        ED = 0
    else:
        EU = P + WU
        if WL >= C * LM:
            EL = (EP - EU) * WL / LM
            ED = 0
        else:
            EL = WL
            ED = C * (EP - EU) - EL
    # 流域总蒸发
    ET = EU + EL + ED

    #### 产流计算
    WMM = (1 + B) * WM / (1 - IM)
    A = WMM * (1 - np.power(1 - W / WM, 1 / (1 + B)))
    PE = P - EP
    if PE < 1e-4:
        R   = 0
        RIM = 0
    else:
        if A + PE <= WMM:
            R = PE + W - WM + WM * np.power(1 - (A + PE) / WMM, 1 + B)
        else:
            R = PE + W - WM
        RIM = PE * IM
    # 计算下一时段初的土壤含水量
    WU = WU + P - EU - R
    WL = WL - EL
    WD = np.max([0, WD - ED])

    if WU > UM:
        WL = WL + WU - UM
        WU = UM
    if WL > LM:
        WD = WD + WL - LM
        WL = LM
    DM = WM - UM - LM
    if WD > DM:
        WD = DM
    W = WU + WL + WD

    #### 分水源计算
    

    final_stat = [EU, EL, ED, ET, WU, WL, WD, W, PE, R, RIM]
    return final_stat